# Lesson 9: Dipoles fitting

Neurocampus course "Signals of the whole brain"

Daria Kleeva

dkleeva@gmail.com

May 13th, 2026


In [ ]:
# !pip3 install nilearn

In [ ]:
import mne
from mne.datasets import sample
import numpy as np
from os import path as op
from mne.forward import make_forward_dipole
from mne.evoked import combine_evoked
from mne.simulation import simulate_evoked
from matplotlib import pyplot as plt
from nilearn.plotting import plot_anat
from nilearn.datasets import load_mni152_template

## Dipoles orientations

In [ ]:
data_path = sample.data_path()
meg_path = data_path / 'MEG' / 'sample'
evokeds = mne.read_evokeds(meg_path / 'sample_audvis-ave.fif')
left_auditory = evokeds[0].apply_baseline()
fwd = mne.read_forward_solution(
    meg_path / 'sample_audvis-meg-eeg-oct-6-fwd.fif')
mne.convert_forward_solution(fwd, surf_ori=True, copy=False)
noise_cov = mne.read_cov(meg_path / 'sample_audvis-cov.fif')
subject = 'sample'
subjects_dir = data_path / 'subjects'
trans_fname = meg_path / 'sample_audvis_raw-trans.fif'

In [ ]:
lh = fwd['src'][0]
verts = lh['rr'] 
tris = lh['tris']
dip_pos = lh['rr'][lh['vertno']] 
dip_ori = lh['nn'][lh['vertno']]
dip_len = len(dip_pos)
dip_times = [0]
white = (1.0, 1.0, 1.0)  
actual_amp = np.ones(dip_len) 
actual_gof = np.ones(dip_len)  
dipoles = mne.Dipole(dip_times, dip_pos, actual_amp, dip_ori, actual_gof)
trans = mne.read_trans(trans_fname)

fig = mne.viz.create_3d_figure(size=(600, 400), bgcolor=white)
coord_frame = 'mri'

mne.viz.plot_alignment(
    subject=subject, subjects_dir=subjects_dir, trans=trans, surfaces='white',
    coord_frame=coord_frame, fig=fig)

mne.viz.plot_dipole_locations(
    dipoles=dipoles, trans=trans, mode='sphere', subject=subject,
    subjects_dir=subjects_dir, coord_frame=coord_frame, scale=7e-4, fig=fig)

mne.viz.set_3d_view(figure=fig, azimuth=180, distance=0.25)

In [ ]:
fig = mne.viz.create_3d_figure(size=(600, 400))


mne.viz.plot_alignment(
    subject=subject, subjects_dir=subjects_dir, trans=trans,
    surfaces='white', coord_frame='head', fig=fig)


mne.viz.plot_dipole_locations(
    dipoles=dipoles, trans=trans, mode='arrow', subject=subject,
    subjects_dir=subjects_dir, coord_frame='head', scale=7e-4, fig=fig)

mne.viz.set_3d_view(figure=fig, azimuth=180, distance=0.1)

In [ ]:
fig = mne.viz.create_3d_figure(size=(600, 400))

mne.viz.plot_alignment(
    subject=subject, subjects_dir=subjects_dir, trans=trans,
    surfaces='white', coord_frame='head', fig=fig)

mne.viz.plot_alignment(
    subject=subject, subjects_dir=subjects_dir, trans=trans, fwd=fwd,
    surfaces='white', coord_frame='head', fig=fig)

mne.viz.set_3d_view(figure=fig, azimuth=180, distance=0.1)

## Process evoked data

In [ ]:
data_path = mne.datasets.sample.data_path()
subjects_dir = op.join(data_path, 'subjects')
fname_ave = op.join(data_path, 'MEG', 'sample', 'sample_audvis-ave.fif')
fname_cov = op.join(data_path, 'MEG', 'sample', 'sample_audvis-cov.fif')
fname_bem = op.join(subjects_dir, 'sample', 'bem', 'sample-5120-bem-sol.fif')
fname_trans = op.join(data_path, 'MEG', 'sample',
                      'sample_audvis_raw-trans.fif')
fname_surf_lh = op.join(subjects_dir, 'sample', 'surf', 'lh.white')
from mne.beamformer import rap_music
from mne.viz import plot_dipole_locations, plot_dipole_amplitudes
from mne.datasets import sample

In [ ]:
evoked = mne.read_evokeds(fname_ave, condition='Right Auditory',
                          baseline=(None, 0))

In [ ]:
evoked

In [ ]:
fig = evoked.plot_joint()

In [ ]:
evoked.pick_types(meg=True, eeg=False)
evoked_full = evoked.copy()
evoked.crop(0.07, 0.08)

In [ ]:
evoked.crop(0.07, 0.08)

In [ ]:
fig = evoked.plot_joint()

## Standard dipole fitting

In [ ]:
dip = mne.fit_dipole(evoked, fname_cov, fname_bem, fname_trans)

In [ ]:
dip

In [ ]:
dip[0]

In [ ]:
fig= dip[0].plot_locations(fname_trans, 'sample', subjects_dir, mode='orthoview')
# fig= dip[0].plot_locations(fname_trans, 'sample', subjects_dir, mode='outlines')


In [ ]:
evoked.pick_types(meg='mag', eeg=False)
dip = mne.fit_dipole(evoked, fname_cov, fname_bem, fname_trans)
fig= dip[0].plot_locations(fname_trans, 'sample', subjects_dir, mode='orthoview')

In [ ]:
evoked = mne.read_evokeds(fname_ave, condition='Right Auditory',
                          baseline=(None, 0))
evoked.pick_types(meg='grad', eeg=False)
evoked_full = evoked.copy()
evoked.crop(0.07, 0.08)
dip = mne.fit_dipole(evoked, fname_cov, fname_bem, fname_trans)
fig= dip[0].plot_locations(fname_trans, 'sample', subjects_dir, mode='orthoview')


## Prediction VS Observation

In [ ]:
evoked = mne.read_evokeds(fname_ave, condition='Right Auditory',
                          baseline=(None, 0))
evoked.pick_types(meg=True, eeg=False)
evoked_full = evoked.copy()
evoked.crop(0.07, 0.08)
dip = mne.fit_dipole(evoked, fname_cov, fname_bem, fname_trans)

In [ ]:
dip = dip[0]
dip

In [ ]:
fwd, stc = make_forward_dipole(dip, fname_bem, evoked.info, fname_trans)
pred_evoked = simulate_evoked(fwd, stc, evoked.info, cov=None, nave=np.inf)


In [ ]:
pred_evoked

In [ ]:
fig = pred_evoked.plot_joint()

In [ ]:
best_idx = np.argmax(dip.gof)
best_time = dip.times[best_idx]

In [ ]:
vmin, vmax = -400, 400 

plot_params = dict(times=best_time, ch_type='mag', outlines='head',
                   colorbar=False)
evoked.plot_topomap(time_format='Measured field', **plot_params)

pred_evoked.plot_topomap(time_format='Predicted field',
                         **plot_params)

diff = combine_evoked([evoked, pred_evoked], weights=[1, -1])
plot_params['colorbar'] = True
diff.plot_topomap(time_format='Difference', **plot_params)
plt.show()

## Vizualization

In [ ]:
trans = mne.read_trans(fname_trans)
subject = 'sample'
mni_pos = mne.head_to_mni(dip[0].pos, mri_head_t=trans,
                          subject=subject, subjects_dir=subjects_dir)

mri_pos = mne.head_to_mri(dip[0].pos, mri_head_t=trans,
                          subject=subject, subjects_dir=subjects_dir)

t1_fname = op.join(subjects_dir, subject, 'mri', 'T1.mgz')
fig_T1 = plot_anat(t1_fname, cut_coords=mri_pos[0], title='Dipole loc.')

template = load_mni152_template()
fig_template = plot_anat(template, cut_coords=mni_pos[0],
                         title='Dipole loc. (MNI Space)')

## Estimate the time courses

In [ ]:
dip[0].gof

In [ ]:
best_idx = np.argmax(dip[0].gof)
best_time = dip[0].times[best_idx]

In [ ]:
best_time

In [ ]:
dip_fixed = mne.fit_dipole(evoked_full, fname_cov, fname_bem, fname_trans,
                           pos=dip[0].pos[best_idx], ori=dip[0].ori[best_idx])
fig=dip_fixed[0].plot(time_unit='s')

## XFit

In [ ]:
meg_path = data_path / 'MEG' / 'sample'
raw_fname = meg_path / 'sample_audvis_raw.fif'
cov_fname = meg_path / 'sample_audvis-shrunk-cov.fif'
bem_dir = data_path / 'subjects' / 'sample' / 'bem'
bem_fname = bem_dir / 'sample-5120-5120-5120-bem-sol.fif'

In [ ]:
raw = mne.io.read_raw_fif(raw_fname)
raw = raw.pick_types(meg=True, eog=True, stim=True)

events = mne.find_events(raw)
event_id = dict(right=1, left=2)
epochs = mne.Epochs(raw, events, event_id,
                    tmin=-0.1, tmax=0.3, baseline=(None, 0),
                    reject=dict(mag=4e-12, grad=4000e-13, eog=150e-6))
epochs.load_data()

epochs.pick('mag')
info=epochs.info

cov = mne.read_cov(cov_fname)
bem = mne.read_bem_solution(bem_fname)

evoked_left = epochs['left'].average()
evoked_right = epochs['right'].average()

In [ ]:
fig = evoked_left.plot_joint()
fig = evoked_right.plot_joint()

In [ ]:
info

In [ ]:
picks_left = mne.read_vectorview_selection('Left', info=info)
picks_left = [ch for ch in picks_left if ch in evoked_left.info['ch_names']]
evoked_fit_left = evoked_left.copy().crop(0.08, 0.08)
evoked_fit_left.pick_channels(picks_left)
cov_fit_left = cov.copy().pick_channels(picks_left)

picks_right = mne.read_vectorview_selection('Right', info=info)
picks_right = [ch for ch in picks_right if ch in evoked_right.info['ch_names']]
evoked_fit_right = evoked_right.copy().crop(0.08, 0.08)
evoked_fit_right.pick_channels(picks_right)
cov_fit_right = cov.copy().pick_channels(picks_right)

In [ ]:
picks_left

In [ ]:
fig = evoked_fit_left.plot_sensors()
fig = evoked_fit_right.plot_sensors()

In [ ]:
fig = evoked_fit_left.plot_topomap([evoked_fit_left.times[0]])
fig = evoked_fit_right.plot_topomap([evoked_fit_left.times[0]])

In [ ]:
dip_left, _ = mne.fit_dipole(evoked_fit_left, cov_fit_left, bem)
dip_right, _ = mne.fit_dipole(evoked_fit_right, cov_fit_right, bem)

In [ ]:
fig= dip_left[0].plot_locations(fname_trans, 'sample', subjects_dir, mode='orthoview')

In [ ]:
fig= dip_right[0].plot_locations(fname_trans, 'sample', subjects_dir, mode='orthoview')

In [ ]:
best_idx = np.argmax(dip_left[0].gof)
best_time = dip_left[0].times[best_idx]
dip_fixed_left = mne.fit_dipole(evoked_full, fname_cov, fname_bem, fname_trans,
                           pos=dip_left[0].pos[best_idx], ori=dip_left[0].ori[best_idx])
best_idx = np.argmax(dip_right[0].gof)
best_time = dip_right[0].times[best_idx]
dip_fixed_right = mne.fit_dipole(evoked_full, fname_cov, fname_bem, fname_trans,
                           pos=dip_right[0].pos[best_idx], ori=dip_right[0].ori[best_idx])
fig=dip_fixed_left[0].plot(time_unit='s')
fig=dip_fixed_right[0].plot(time_unit='s')

In [ ]:
plt.plot(dip_fixed_left[0].times,dip_fixed_left[0].data[0,:])
plt.plot(dip_fixed_right[0].times,dip_fixed_right[0].data[0,:])

Be careful

In [ ]:
picks_left = mne.read_vectorview_selection('Left-occipital', info=info)
picks_left = [ch for ch in picks_left if ch in evoked_left.info['ch_names']]
evoked_fit_left = evoked_left.copy().crop(0.08, 0.08)
evoked_fit_left.pick_channels(picks_left)
cov_fit_left = cov.copy().pick_channels(picks_left)

picks_right = mne.read_vectorview_selection('Right-occipital', info=info)
picks_right = [ch for ch in picks_right if ch in evoked_right.info['ch_names']]
evoked_fit_right = evoked_right.copy().crop(0.08, 0.08)
evoked_fit_right.pick_channels(picks_right)
cov_fit_right = cov.copy().pick_channels(picks_right)

In [ ]:
fig = evoked_fit_left.plot_sensors()
fig = evoked_fit_right.plot_sensors()

In [ ]:
fig = evoked_fit_left.plot_topomap([evoked_fit_left.times[0]])
fig = evoked_fit_right.plot_topomap([evoked_fit_left.times[0]])

In [ ]:
dip_left, _ = mne.fit_dipole(evoked_fit_left, cov_fit_left, bem)
dip_right, _ = mne.fit_dipole(evoked_fit_right, cov_fit_right, bem)

In [ ]:
fig= dip_left[0].plot_locations(fname_trans, 'sample', subjects_dir, mode='orthoview')

In [ ]:
fig= dip_right[0].plot_locations(fname_trans, 'sample', subjects_dir, mode='orthoview')

In [ ]:
best_idx = np.argmax(dip_left[0].gof)
best_time = dip_left[0].times[best_idx]
dip_fixed_left = mne.fit_dipole(evoked_full, fname_cov, fname_bem, fname_trans,
                           pos=dip_left[0].pos[best_idx], ori=dip_left[0].ori[best_idx])
best_idx = np.argmax(dip_right[0].gof)
best_time = dip_right[0].times[best_idx]
dip_fixed_right = mne.fit_dipole(evoked_full, fname_cov, fname_bem, fname_trans,
                           pos=dip_right[0].pos[best_idx], ori=dip_right[0].ori[best_idx])
fig=dip_fixed_left[0].plot(time_unit='s')
fig=dip_fixed_right[0].plot(time_unit='s')

In [ ]:
picks_vert = mne.read_vectorview_selection('Vertex', info=info)
picks_vert = [ch for ch in picks_vert if ch in evoked_left.info['ch_names']]
evoked_fit_vert = evoked_left.copy().crop(0.08, 0.08)
evoked_fit_vert.pick_channels(picks_vert)
cov_fit_vert = cov.copy().pick_channels(picks_vert)


In [ ]:
fig = evoked_fit_vert.plot_sensors()

In [ ]:
fig = evoked_fit_vert.plot_topomap([evoked_fit_left.times[0]])

In [ ]:
dip_vert, _ = mne.fit_dipole(evoked_fit_vert, cov_fit_vert, bem)

In [ ]:
fig= dip_vert[0].plot_locations(fname_trans, 'sample', subjects_dir, mode='orthoview')

In [ ]:
best_idx = np.argmax(dip_vert[0].gof)
best_time = dip_vert[0].times[best_idx]
dip_fixed_vert = mne.fit_dipole(evoked_full, fname_cov, fname_bem, fname_trans,
                           pos=dip_vert[0].pos[best_idx], ori=dip_vert[0].ori[best_idx])
fig=dip_fixed_vert[0].plot(time_unit='s')

## RAP MUSIC

In [ ]:
data_path = sample.data_path()
subjects_dir = str(data_path) + '/subjects'
fwd_fname = str(data_path) + '/MEG/sample/sample_audvis-meg-eeg-oct-6-fwd.fif'
evoked_fname = str(data_path) + '/MEG/sample/sample_audvis-ave.fif'
cov_fname = str(data_path) + '/MEG/sample/sample_audvis-cov.fif'

condition = 'Right Auditory'
evoked = mne.read_evokeds(evoked_fname, condition=condition,
                          baseline=(None, 0))
evoked.crop(tmin=0.05, tmax=0.15)

evoked.pick_types(meg=True, eeg=False)

forward = mne.read_forward_solution(fwd_fname)

noise_cov = mne.read_cov(cov_fname)

In [ ]:
dipoles, residual = rap_music(evoked, forward, noise_cov, n_dipoles=2,
                              return_residual=True, verbose=True)

In [ ]:
dipoles

In [ ]:
trans = forward['mri_head_t']
fig = plot_dipole_locations(dipoles[0], trans, 'sample', subjects_dir=subjects_dir)

In [ ]:
fig = evoked.plot_topomap()

In [ ]:
fwd_dip, stc = make_forward_dipole(dipoles, bem, evoked.info, trans)
print(stc)

pred_evoked = simulate_evoked(fwd_dip, stc, evoked.info, cov=None, nave=np.inf)
pred_evoked.plot_topomap()
diff = combine_evoked([evoked, pred_evoked], weights=[1, -1])
fig = diff.plot_topomap()

In [ ]:
fig = plot_dipole_amplitudes(dipoles)
